# Suffix-Balanced SMC on Google Colab

This notebook runs only the independent `suffix_balanced_smc` pipeline. It stores one resumable directory per seed in Google Drive and never launches direct sampling, mixture MCMC, TPS, MBAR, or another SMC implementation.

In [ ]:
# EDIT PARAMETERS HERE: review this single cell before a long run.
ALGORITHMS_TO_RUN = ["suffix_balanced_smc"]
SEEDS = [200, 201, 202]
NUM_PARTICLES = 256
MAX_NEW_TOKENS = 100
LAMBDAS = "0,-0.25,-0.5,-0.75,-1.0,-1.25,-1.5,-1.75,-2.0,-2.25,-2.5,-2.75,-3.0,-3.25,-3.5,-3.75,-4.0,-4.25,-4.5,-4.75,-5.0"
MCMC_STEPS_PER_LEVEL = 10
ESS_THRESHOLD = 0.5

SUFFIX_MODE = "stochastic"
SUFFIX_SIZES = "1,2,4,8,16,32,64,100"
SUFFIX_RHOS = "0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04"
SUFFIX_SIZE_SCHEDULE = None
SUFFIX_RHO_SCHEDULE = None

BALANCING_FUNCTION = "barker"  # barker | metropolis | sqrt
PHI_MIN = None                    # required for sqrt
PHI_MAX = None                    # required for sqrt
SQRT_SCALE = None                 # None selects the certified automatic scale
PROPOSAL_TEMPERATURE = 1.0        # must remain exactly 1.0
BATCH_SIZE = 32
TAIL_THRESHOLDS = "8,10,12"
DEVICE = "cuda"
EXPERIMENT_LABEL = "suffix_balanced_smc_lambda_minus5"
SKIP_COMPLETED_RUNS = True

In [ ]:
# Mount Drive, then clone or update the repository.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main"
REPO_DIR = Path('/content/rea_of_llms_supplementary_code-main')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('cwd =', Path.cwd())

In [ ]:
# Install the repository dependencies in the current Colab runtime.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Dependencies installed.')

In [ ]:
# Verify that Colab really exposes the requested GPU.
import subprocess, torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before continuing.'
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# Validate the mathematical and scheduling assumptions before spending compute.
import math
assert ALGORITHMS_TO_RUN == ['suffix_balanced_smc']
assert len(SEEDS) == len(set(SEEDS)), 'Seeds must be distinct.'
lambda_values = [float(x) for x in LAMBDAS.split(',')]
assert math.isclose(lambda_values[0], 0.0)
assert all(b <= a for a, b in zip(lambda_values, lambda_values[1:])), 'Lambdas must be non-increasing.'
assert PROPOSAL_TEMPERATURE == 1.0, 'Exact LM cancellation requires temperature one.'
suffix_sizes = [int(x) for x in SUFFIX_SIZES.split(',')]
assert len(suffix_sizes) == len(set(suffix_sizes))
assert all(1 <= size <= MAX_NEW_TOKENS for size in suffix_sizes)
if SUFFIX_MODE == 'stochastic':
    rho_rows = ([SUFFIX_RHOS] if SUFFIX_RHO_SCHEDULE is None else SUFFIX_RHO_SCHEDULE.split(';'))
    for rho_row in rho_rows:
        suffix_rhos = [float(x) for x in rho_row.split(',')]
        assert len(suffix_rhos) == len(suffix_sizes)
        assert all(rho >= 0 for rho in suffix_rhos) and math.isclose(sum(suffix_rhos), 1.0)
        assert MAX_NEW_TOKENS in suffix_sizes
        assert suffix_rhos[suffix_sizes.index(MAX_NEW_TOKENS)] > 0, 'm=T needs positive mass for global irreducibility.'
else:
    assert SUFFIX_SIZE_SCHEDULE is not None, 'Deterministic mode requires SUFFIX_SIZE_SCHEDULE.'
if BALANCING_FUNCTION == 'sqrt':
    assert PHI_MIN is not None and PHI_MAX is not None and PHI_MIN <= PHI_MAX
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_suffix_balanced_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Validated. Outputs:', OUTPUT_ROOT)

In [ ]:
# Run independent seeds. CLI progress reports each completed SMC level.
import json, subprocess, sys
from tqdm.auto import tqdm

for seed in tqdm(SEEDS, desc='Independent SMC seeds'):
    seed_dir = OUTPUT_ROOT / EXPERIMENT_LABEL / f'seed_{seed}'
    summary_path = seed_dir / 'summary.json'
    if SKIP_COMPLETED_RUNS and summary_path.exists():
        old = json.loads(summary_path.read_text())
        if (old.get('algorithm') == 'suffix_balanced_smc' and old.get('config', {}).get('seed') == seed and old.get('lambdas') == lambda_values):
            print(f'seed {seed}: complete, skipping')
            continue
    command = [
        sys.executable, 'suffix_balanced_smc_experiment/run_experiment.py',
        '--device', DEVICE, '--num-particles', str(NUM_PARTICLES),
        '--max-new-tokens', str(MAX_NEW_TOKENS), '--lambdas', LAMBDAS,
        '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
        '--ess-threshold', str(ESS_THRESHOLD), '--suffix-mode', SUFFIX_MODE,
        '--suffix-sizes', SUFFIX_SIZES, '--balancing-function', BALANCING_FUNCTION,
        '--proposal-temperature', str(PROPOSAL_TEMPERATURE),
        '--batch-size', str(BATCH_SIZE), '--tail-thresholds', TAIL_THRESHOLDS,
        '--implementation', 'parallel', '--seed', str(seed),
        '--output-root', str(OUTPUT_ROOT), '--experiment-label', EXPERIMENT_LABEL,
    ]
    if SUFFIX_MODE == 'stochastic':
        if SUFFIX_RHO_SCHEDULE is None: command += ['--suffix-rhos', SUFFIX_RHOS]
        else: command += ['--suffix-rho-schedule', SUFFIX_RHO_SCHEDULE]
    else:
        command += ['--suffix-size-schedule', SUFFIX_SIZE_SCHEDULE]
    if PHI_MIN is not None: command += ['--phi-min', str(PHI_MIN)]
    if PHI_MAX is not None: command += ['--phi-max', str(PHI_MAX)]
    if SQRT_SCALE is not None: command += ['--sqrt-scale', str(SQRT_SCALE)]
    print('Running seed', seed)
    subprocess.run(command, check=True)

In [ ]:
# Load and validate every result before plotting.
import json, pandas as pd, numpy as np
payloads, sample_frames = {}, {}
for seed in SEEDS:
    seed_dir = OUTPUT_ROOT / EXPERIMENT_LABEL / f'seed_{seed}'
    payload = json.loads((seed_dir / 'summary.json').read_text())
    samples = pd.read_json(seed_dir / 'samples.jsonl', lines=True)
    assert payload['algorithm'] == 'suffix_balanced_smc'
    assert payload['implementation'] == 'parallel'
    assert payload['config']['seed'] == seed
    assert np.allclose(payload['lambdas'], lambda_values)
    assert len(samples) == NUM_PARTICLES
    assert samples['completion_token_ids'].map(len).eq(MAX_NEW_TOKENS).all()
    assert np.isclose(samples['normalized_weight'].sum(), 1.0)
    payloads[seed], sample_frames[seed] = payload, samples
summary_table = pd.DataFrame([{
    'seed': seed, 'weighted_mean_ari': p['weighted_mean_ari'],
    'unweighted_mean_ari': p['unweighted_mean_ari'], 'max_ari': p['max_ari'],
    'final_ess_over_n': p['final_ess_over_n'],
    'acceptance_rate': p['mean_mcmc_acceptance_rate'],
    'resampling_steps': p['resampling_steps'],
    'distinct_root_ancestors': p['final_distinct_root_ancestors'],
    'runtime_seconds': p['runtime_total_seconds'],
} for seed, p in payloads.items()])
display(summary_table)

In [ ]:
# Core SMC diagnostics by lambda.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for seed, payload in payloads.items():
    levels = pd.DataFrame(payload['diagnostics']['level_diagnostics'])
    axes[0,0].plot(levels['lambda'], levels['weighted_mean_ari_before_resampling'], marker='o', label=str(seed))
    axes[0,1].plot(levels['lambda'], levels['ess_over_n_before_resampling'], marker='o', label=str(seed))
    resampled = levels[levels['resampled']]
    axes[0,1].scatter(resampled['lambda'], resampled['ess_over_n_before_resampling'], marker='x', s=90)
    mutation = levels.iloc[1:]
    axes[1,0].plot(mutation['lambda'], mutation['acceptance_rate'], marker='o', label=f'total {seed}')
    axes[1,0].plot(mutation['lambda'], mutation['mean_symmetrization_probability'], linestyle='--', label=f'a_sym {seed}')
    axes[1,0].plot(mutation['lambda'], mutation['mean_balancing_component'], linestyle=':', label=f'g {seed}')
    samples = sample_frames[seed].sort_values('raw_ari')
    axes[1,1].step(samples['raw_ari'], samples['normalized_weight'].cumsum(), where='post', label=str(seed))
axes[0,0].set(title='Weighted mean raw ARI by level', xlabel='lambda', ylabel='weighted mean ARI')
axes[0,1].axhline(ESS_THRESHOLD, color='red', linestyle='--', label='ESS threshold')
axes[0,1].set(title='ESS and resampling markers', xlabel='lambda', ylabel='ESS/N')
axes[1,0].set(title='Acceptance components', xlabel='lambda after transition', ylabel='mean probability/component')
axes[1,1].set(title='Final weighted ARI ECDF', xlabel='raw ARI', ylabel='weighted cumulative mass')
for ax in axes.flat: ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Across-seed comparison. The boxplot is explicitly unweighted empirical ARI.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(summary_table))
axes[0].bar(x - 0.18, summary_table['weighted_mean_ari'], 0.36, label='weighted mean')
axes[0].bar(x + 0.18, summary_table['max_ari'], 0.36, label='raw maximum')
axes[0].set_xticks(x, summary_table['seed'].astype(str)); axes[0].set_xlabel('seed'); axes[0].set_ylabel('raw ARI')
axes[0].set_title('Final quality by independent seed'); axes[0].legend()
axes[1].boxplot([sample_frames[s]['raw_ari'] for s in SEEDS], tick_labels=[str(s) for s in SEEDS], showmeans=True)
axes[1].set(title='Unweighted empirical final-particle ARI', xlabel='seed', ylabel='raw ARI')
for ax in axes: ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

In [ ]:
# Inspect the highest-ARI completions with their final SMC weights and ancestry.
pd.set_option('display.max_colwidth', None)
all_samples = pd.concat([frame.assign(seed=seed) for seed, frame in sample_frames.items()], ignore_index=True)
highest = all_samples.nlargest(20, 'raw_ari')[[
    'seed', 'particle_id', 'raw_ari', 'phi', 'normalized_weight',
    'ancestor_id', 'last_suffix_size', 'completion'
]]
display(highest)